# Information Retrieval with RAG Pipeline

by Mickey Choy and Yuheng Ouyang

## 0. Imports

In [1]:
import pandas as pd
from IPython.display import display, Markdown

import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

import warnings
warnings.filterwarnings('ignore')

from src.semantic import load_faiss

from src.rag_pipeline import (
    build_llm,
    build_context,
    build_prompt_template,
    build_semantic_rag_chain,
    build_hybrid_rag_chain
)

## 1. LLM prompt template design

## 1.2. Prompt variants

We defined three versions of the system prompt: one as a safe baseline, one expected to have a more helpful recommendation style, and the last expected to be the easiest to inspect. 

In [2]:
SYSTEM_PROMPT_V1 = """
You are a helpful Amazon shopping assistant.
Answer the user's question using ONLY the provided product context.
If the context is insufficient, say that the available product information is not enough to answer confidently.
Be concise and practical.
"""

SYSTEM_PROMPT_V2 = """
You are a helpful Amazon shopping assistant.
Use ONLY the provided product context to answer the question.
Give a short recommendation-oriented answer that highlights the most relevant products and why they match the query.
If possible, mention product titles or ASINs.
If the context is insufficient, say so clearly.
"""

SYSTEM_PROMPT_V3 = """
You are a helpful Amazon shopping assistant.
Answer using ONLY the provided context. Do not use outside knowledge.
Write 2-4 bullet points covering:
- best match(es)
- why they are relevant
- any important limitation or uncertainty
If the retrieved context does not clearly support an answer, explicitly say that.
"""

prompt_variants = {
    "V1": SYSTEM_PROMPT_V1,
    "V2": SYSTEM_PROMPT_V2,
    "V3": SYSTEM_PROMPT_V3,
}

## 1.3. Test queries

We created three queries to test the performance of the prompts: one keyword-based query, one more descriptive, and the last requires the most logical understanding. 

In [3]:
test_queries = [
    "refrigerator water filter",
    "something to keep drinks cold in a dorm room",
    "energy-efficient refrigerator for a family of four",
]

## 1.4. Results and comparison

In [4]:
_, docs = load_faiss("../semantic_index")

results = []

for variant_name, system_prompt in prompt_variants.items():
    rag_chain = build_semantic_rag_chain(
        docs,
        k=5,
        system_prompt=system_prompt
    )
    
    for query in test_queries:
        answer = rag_chain.invoke(query)
        results.append({
            "query": query,
            "prompt_variant": variant_name,
            "answer": answer,
        })

results_df = pd.DataFrame(results)
results_df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,query,prompt_variant,answer
0,refrigerator water filter,V1,"Based on the provided context, the product tha..."
1,something to keep drinks cold in a dorm room,V1,"Based on the available product information, th..."
2,energy-efficient refrigerator for a family of ...,V1,"Based on the provided product context, I found..."
3,refrigerator water filter,V2,"Based on the provided context, I would recomme..."
4,something to keep drinks cold in a dorm room,V2,"Based on your query, I recommend the COOLLIFE ..."
5,energy-efficient refrigerator for a family of ...,V2,For an energy-efficient refrigerator suitable ...
6,refrigerator water filter,V3,"Based on the provided context, here are some r..."
7,something to keep drinks cold in a dorm room,V3,Based on the provided context:\n\n* The best m...
8,energy-efficient refrigerator for a family of ...,V3,"Unfortunately, the retrieved context does not ..."


In [5]:
for query in test_queries:
    display(Markdown(f"## Query: \"{query}\""))
    
    subset = results_df[results_df["query"] == query]
    
    for _, row in subset.iterrows():
        display(Markdown(f"### {row['prompt_variant']}"))
        display(Markdown(row["answer"]))

## Query: "refrigerator water filter"

### V1

Based on the provided context, the product that matches "refrigerator water filter" is:

- GE refrigerator water filter (B009PCI2JU)
- Fridge water filter (B00UB38V2A)

### V2

Based on the provided context, I would recommend the following products for a refrigerator water filter:

1. **B00UB38V2A**: This product has a 5.0/5 rating and a verified purchase, indicating it works well for

...

Based on the provided product context, I found reviews for two refrigerators. Unfortunately, neither of them specifically mentions being energy-efficient and catering to a family of four.

However, the review for ASIN: B00O2CF24Q (Bang

### V2

For an energy-efficient refrigerator suitable for a family of four, I recommend considering compact or mini options like the KRIB BLING 3.5 Cu.ft Retro Mini Fridge or the FRIGIDAIRE Portable 10L Mini Fridge. These

### V3

Unfortunately, the retrieved context does not clearly support an answer to the question about an energy-efficient refrigerator for a family of four.

We compared three prompt variants while keeping the retriever and model fixed. V1 produced concise but sometimes overly generic answers, while V2 generated more natural recommendation-style responses but could sound too confident when retrieval quality was weak. V3 performed best overall because it balanced usefulness and grounding: its structured bullet format made relevant products easier to inspect and helped surface uncertainty more clearly on difficult queries. Based on this comparison, V3 was selected as the final prompt for the RAG system.

## 2. Qualitative evaluation

In [6]:
queries = [
    "stainless steel dishwasher",
    "refrigerator water filter",
    "gas range 30 inch",
    "something to keep drinks cold in a dorm room",
    "appliance for washing dishes quietly at night",
    "replacement part to stop a dishwasher from leaking at the bottom",
    "small machine for drying clothes in an apartment",
    "best dishwasher for a small apartment under $800",
    "energy-efficient refrigerator for a family of four",
    "reliable stove for frequent home cooking that is easy to clean",
]

In [7]:
hybrid_chain = build_hybrid_rag_chain(docs=docs, k=5)

results = []

for query in queries:
    answer = hybrid_chain.invoke(query)

    results.append({
        "query": query,
        "answer": answer,
    })

results_df = pd.DataFrame(results)
results_df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,query,answer
0,stainless steel dishwasher,"Based on the provided context, here are 3 bull..."
1,refrigerator water filter,"Based on the provided context, here are 3 bull..."
2,gas range 30 inch,"Based on the provided context, here are the re..."
3,something to keep drinks cold in a dorm room,"Based on the provided reviews and metadata, th..."
4,appliance for washing dishes quietly at night,"Based on the provided reviews and metadata, he..."
5,replacement part to stop a dishwasher from lea...,Based on the provided context:\n\n* Best Match...
6,small machine for drying clothes in an apartment,"Based on the provided context, here are 3 rele..."
7,best dishwasher for a small apartment under $800,- ASIN: B09MHZ9TBB \n - Why they are relevant...
8,energy-efficient refrigerator for a family of ...,"Based on the provided reviews and metadata, th..."
9,reliable stove for frequent home cooking that ...,"Based on the provided context, here are 3-4 bu..."


In [8]:
for _, row in results_df.iterrows():
    display(Markdown(f"### Query: {row['query']}"))
    display(Markdown(row["answer"]))

### Query: stainless steel dishwasher

Based on the provided context, here are 3 bullet points covering the best match(es) for a stainless steel dishwasher:

* ASIN: B00NN1CAJW - Front Control Dishwasher in White with Stainless Steel Tub: This item matches

### Query: refrigerator water filter

Based on the provided context, here are 3 bullet points covering the best match(es) for a refrigerator water filter:

* **Best Match:** DA97-17376B Refrigerator Water Filter 
   - Why it's relevant: It's a

### Query: gas range 30 inch

...

- ASIN: B09MHZ9TBB 
  - Why they are relevant: VENTRAY Countertop Portable Dishwasher Mini Compact with 5 Washing Programs LED Digital Display is a compact dishwasher suitable for small apartments.
  - Any

### Query: energy-efficient refrigerator for a family of four

Based on the provided reviews and metadata, the best match for an energy-efficient refrigerator for a family of four is uncertain. The reviews do not mention energy efficiency. However, the following points are relevant:

* A refrigerator with a compact design and counter-depth

### Query: reliable stove for frequent home cooking that is easy to clean

Based on the provided context, here are 3-4 bullet points covering the best match(es) for a reliable stove for frequent home cooking that is easy to clean:

* ASIN: B07M8JWVN1 is a good match